In [ ]:
import pandas as pd
import json
import os
from google.colab import drive

# --- 1. IPA Rule Data ---
IPA_RULES = {
  "dialects": {
    "Egyptian": {
      "letters": {
        "ء": { "phoneme": "/ʔ/" }, "أ": { "phoneme": "/ʔ/" }, "ب": { "phoneme": "/b/" },
        "ت": { "phoneme": "/t/" }, "ث": { "phoneme": "/t/" }, "ج": { "phoneme": "/ɡ/" },
        "ح": { "phoneme": "/ħ/" }, "خ": { "phoneme": "/x/" }, "د": { "phoneme": "/d/" },
        "ذ": { "phoneme": "/d/" }, "ر": { "phoneme": "/ɾ/" }, "ز": { "phoneme": "/z/" },
        "س": { "phoneme": "/s/" }, "ش": { "phoneme": "/ʃ/" }, "ص": { "phoneme": "/sˤ/" },
        "ض": { "phoneme": "/dˤ/" }, "ط": { "phoneme": "/tˤ/" }, "ظ": { "phoneme": "/zˤ/" },
        "ع": { "phoneme": "/ʕ/" }, "غ": { "phoneme": "/ɣ/" }, "ف": { "phoneme": "/f/" },
        "ق": { "phoneme": "/ʔ/" }, "ك": { "phoneme": "/k/" }, "ل": { "phoneme": "/l/" },
        "م": { "phoneme": "/m/" }, "ن": { "phoneme": "/n/" }, "ه": { "phoneme": "/h/" },
        "و": { "phoneme": "/w/" }, "ي": { "phoneme": "/j/" }, "ة": { "phoneme": "/a/ or /et/" }
      },
      "vowels": {
        "short": { "َ": "/a/", "ُ": "/u/", "ِ": "/i/" },
        "long": { "ا": "/aː/", "و": "/uː/", "ي": "/iː/" }
      },
      "diacritics": { "ً": "/an/", "ٌ": "/un/", "ٍ": "/in/", "ْ": "", "ّ": "doubled" },
      "phonological_rules": {
        "al_prefix_moon": { "letters": "أ, ب, ج, ح, خ, ع, غ, f, ق, ك, م, ه, و, ي", "ipa": "/il-/" },
        "al_prefix_sun": { "letters": "ت, ث, d, ذ, ر, ز, س, ش, ص, ض, ط, ظ, ل, ن", "ipa": "/i-/" }
      }
    },
    "Levantine Arabic": {
      "letters": { "ء": { "phoneme": "/ʔ/" }, "ث": { "phoneme": "/s/" }, "ج": { "phoneme": "/ʒ/" }, "ذ": { "phoneme": "/z/" }, "ق": { "phoneme": "/ʔ/" }, "ة": { "phoneme": "/a/ or /e/" } },
      "vowels": { "short": { "َ": "/a/", "ُ": "/u/", "ِ": "/i/" } },
      "diacritics": { "ّ": "doubled", "ْ": "" },
      "phonological_rules": { "al_prefix_moon": { "ipa": "/il-/" }, "al_prefix_sun": { "ipa": "/i-/" } }
    },
    "Khaliji Arabic": {
      "letters": { "ء": { "phoneme": "/ʔ/" }, "ث": { "phoneme": "/θ/" }, "ج": { "phoneme": "/dʒ/" }, "ذ": { "phoneme": "/ð/" }, "ق": { "phoneme": "/g/" }, "ة": { "phoneme": "/a/ or /at/" } },
      "vowels": { "short": { "َ": "/a/", "ُ": "/u/", "ِ": "/i/" } },
      "diacritics": { "ّ": "doubled", "ْ": "" },
      "phonological_rules": { "al_prefix_moon": { "ipa": "/al-/" }, "al_prefix_sun": { "ipa": "/a-/" } }
    },
    "Modern Standard Arabic (MSA)": {
      "letters": { "ء": { "phoneme": "/ʔ/" }, "ث": { "phoneme": "/θ/" }, "ج": { "phoneme": "/dʒ/" }, "ذ": { "phoneme": "/ð/" }, "ق": { "phoneme": "/q/" }, "ة": { "phoneme": "/a/ or /at/" } },
      "vowels": { "short": { "َ": "/a/", "ُ": "/u/", "ِ": "/i/" } },
      "diacritics": { "ّ": "doubled", "ْ": "" },
      "phonological_rules": { "al_prefix_moon": { "ipa": "/al-/" }, "al_prefix_sun": { "ipa": "/a-/" } }
    }
  }
}

# Sync all dialects with base Egyptian letters
for d in ["Levantine Arabic", "Khaliji Arabic", "Modern Standard Arabic (MSA)"]:
    base = IPA_RULES["dialects"]["Egyptian"]["letters"].copy()
    base.update(IPA_RULES["dialects"][d]["letters"])
    IPA_RULES["dialects"][d]["letters"] = base
    IPA_RULES["dialects"][d]["vowels"] = IPA_RULES["dialects"]["Egyptian"]["vowels"]
    IPA_RULES["dialects"][d]["diacritics"].update({"ً": "/an/", "ٌ": "/un/", "ٍ": "/in/"})

# --- 2. Helper Functions ---

def get_rules(dialect):
    if not isinstance(dialect, str): return IPA_RULES["dialects"]["Modern Standard Arabic (MSA)"]
    if "Egyptian" in dialect: return IPA_RULES["dialects"]["Egyptian"]
    if "Levantine" in dialect: return IPA_RULES["dialects"]["Levantine Arabic"]
    if "Khaliji" in dialect or "Gulf" in dialect: return IPA_RULES["dialects"]["Khaliji Arabic"]
    return IPA_RULES["dialects"]["Modern Standard Arabic (MSA)"]

def transcribe(text, dialect):
    if pd.isna(text) or text == "": return ""
    rules = get_rules(dialect)
    letters = rules["letters"]
    diacritics = {**rules["vowels"]["short"], **rules["diacritics"]}
    sun_letters = set(rules["phonological_rules"]["al_prefix_sun"].get("letters", "تثدذرزسشصضطظلن").replace(",", ""))

    words = str(text).split()
    results = []

    for word in words:
        out = ""
        i = 0
        while i < len(word):
            char = word[i]

            # 1. Handle "li-l-" (للـ) prefix
            if char == 'ل' and i + 1 < len(word) and word[i+1] == 'ل':
                out += "/l/il-/"
                i += 2
                continue

            # 2. Handle "Al-" (الـ) Prefix
            if char == 'ا' and i + 1 < len(word) and word[i+1] == 'ل':
                if i + 2 < len(word) and word[i+2] in sun_letters:
                    prefix = rules["phonological_rules"]["al_prefix_sun"]["ipa"].strip('/')
                    sun_ph = letters.get(word[i+2], {}).get("phoneme", "").strip('/')
                    out += f"/{prefix}{sun_ph}{sun_ph}/"
                    i += 3
                    continue
                else:
                    out += rules["phonological_rules"]["al_prefix_moon"]["ipa"]
                    i += 2
                    continue

            # 3. Handle Letters
            if char in letters:
                ph = letters[char]["phoneme"]
                if " or " in ph: ph = ph.split(" or ")[0] # Pausal form for ة
                out += ph

            # 4. Handle Diacritics/Shadda
            if i + 1 < len(word) and word[i+1] in diacritics:
                d = word[i+1]
                if d == "ّ":
                    # Double the previous sound segment
                    clean_out = out.strip('/')
                    if clean_out:
                        last_unit = clean_out.split('/')[-1]
                        out += f"/{last_unit}/"
                else:
                    out += diacritics[d]
                i += 1
            i += 1

        # Clean up separators for the word
        clean_word = out.replace("//", "/").strip('/')
        if clean_word:
            results.append("/" + clean_word + "/")

    return " ".join(results)

# --- 3. Main Logic ---

if __name__ == "__main__":
    drive.mount('/content/drive')
    path = input("Enter the full path to your CSV file: ")

    try:
        # Load CSV
        try:
            df = pd.read_csv(path, sep='|', encoding='utf-8')
        except:
            df = pd.read_csv(path, sep='|', encoding='windows-1256')

        # Flexible Column Mapping
        cols = {c.lower().strip(): c for c in df.columns}

        t_col = cols.get('transcript')
        # Check for 'main dialect' or 'dialect'
        d_col = cols.get('main dialect') or cols.get('dialect')
        # Check for 'phoneme' or 'phonemes'
        p_col = cols.get('phoneme') or cols.get('phonemes') or 'Phoneme'

        if not t_col or not d_col:
            print(f"Error: Missing required columns. Found: {list(df.columns)}")
            print("Required: 'Transcript' and ('Main Dialect' or 'Dialect')")
        else:
            print(f"Processing: Dialect from '{d_col}', Transcript from '{t_col}'...")
            df[p_col] = df.apply(lambda r: transcribe(r[t_col], r[d_col]), axis=1)

            # Save back with same format
            df.to_csv(path, index=False, sep='|', encoding='utf-8')
            print(f"Success! Updated {path}")
            print(df[[t_col, p_col]].head())

    except Exception as e:
        print(f"An unexpected error occurred: {e}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Enter the full path to your CSV file: /content/drive/MyDrive/Masters Arabic TTS/Organized/Egyptian/Amr Abdeen/Topic_001/metadata.csv
Processing: Dialect from 'Main Dialect', Transcript from 'Transcript'...
Success! Updated /content/drive/MyDrive/Masters Arabic TTS/Organized/Egyptian/Amr Abdeen/Topic_001/metadata.csv
                                          Transcript  \
0  اللي متأكدين منه أنه من سكان نيوزيلندا الأصليي...   
1                          وطوله كان حوالي 170 سنتين   
2  الشرطة بالقرب من مسرح الجريمة حققت مع واحد اسم...   
3          بتنطبق عليه المواصفات اللي الأولاد قالوها   
4  وعرفوا كمان أنه كان عنده حربة وطاقيّة صوف بس ض...   

                                             Phoneme  
0  /i-ll/j/ /m/t/ʔ/k/d/j/n/ /m/n/h/ /ʔ/n/h/ /m/n/...  
1         /w/tˤ/w/l/h/ /k/n/ /ħ/w/il-/j/ /s/n/t/j/n/  
2  /i-ʃʃ/ɾ/tˤ/a/ /b/il-/ʔ/ɾ/b/ /m/n/ /m/s/ɾ/ħ/ /i..

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Iterative Code

In [ ]:
import pandas as pd
import json
import os
from google.colab import drive

# --- 1. IPA Rule Data ---
IPA_RULES = {
  "dialects": {
    "Egyptian": {
      "letters": {
        "ء": { "phoneme": "/ʔ/" }, "أ": { "phoneme": "/ʔ/" }, "ب": { "phoneme": "/b/" },
        "ت": { "phoneme": "/t/" }, "ث": { "phoneme": "/t/" }, "ج": { "phoneme": "/ɡ/" },
        "ح": { "phoneme": "/ħ/" }, "خ": { "phoneme": "/x/" }, "د": { "phoneme": "/d/" },
        "ذ": { "phoneme": "/d/" }, "ر": { "phoneme": "/ɾ/" }, "ز": { "phoneme": "/z/" },
        "س": { "phoneme": "/s/" }, "ش": { "phoneme": "/ʃ/" }, "ص": { "phoneme": "/sˤ/" },
        "ض": { "phoneme": "/dˤ/" }, "ط": { "phoneme": "/tˤ/" }, "ظ": { "phoneme": "/zˤ/" },
        "ع": { "phoneme": "/ʕ/" }, "غ": { "phoneme": "/ɣ/" }, "ف": { "phoneme": "/f/" },
        "ق": { "phoneme": "/ʔ/" }, "ك": { "phoneme": "/k/" }, "ل": { "phoneme": "/l/" },
        "م": { "phoneme": "/m/" }, "ن": { "phoneme": "/n/" }, "ه": { "phoneme": "/h/" },
        "و": { "phoneme": "/w/" }, "ي": { "phoneme": "/j/" }, "ة": { "phoneme": "/a/ or /et/" }
      },
      "vowels": {
        "short": { "َ": "/a/", "ُ": "/u/", "ِ": "/i/" },
        "long": { "ا": "/aː/", "و": "/uː/", "ي": "/iː/" }
      },
      "diacritics": { "ً": "/an/", "ٌ": "/un/", "ٍ": "/in/", "ْ": "", "ّ": "doubled" },
      "phonological_rules": {
        "al_prefix_moon": { "letters": "أ, ب, ج, ح, خ, ع, غ, f, ق, ك, م, ه, و, ي", "ipa": "/il-/" },
        "al_prefix_sun": { "letters": "ت, ث, d, ذ, ر, ز, س, ش, ص, ض, ط, ظ, ل, ن", "ipa": "/i-/" }
      }
    },
    "Levantine Arabic": {
      "letters": { "ء": { "phoneme": "/ʔ/" }, "ث": { "phoneme": "/s/" }, "ج": { "phoneme": "/ʒ/" }, "ذ": { "phoneme": "/z/" }, "ق": { "phoneme": "/ʔ/" }, "ة": { "phoneme": "/a/ or /e/" } },
      "vowels": { "short": { "َ": "/a/", "ُ": "/u/", "ِ": "/i/" } },
      "diacritics": { "ّ": "doubled", "ْ": "" },
      "phonological_rules": { "al_prefix_moon": { "ipa": "/il-/" }, "al_prefix_sun": { "ipa": "/i-/" } }
    },
    "Khaliji Arabic": {
      "letters": { "ء": { "phoneme": "/ʔ/" }, "ث": { "phoneme": "/θ/" }, "ج": { "phoneme": "/dʒ/" }, "ذ": { "phoneme": "/ð/" }, "ق": { "phoneme": "/g/" }, "ة": { "phoneme": "/a/ or /at/" } },
      "vowels": { "short": { "َ": "/a/", "ُ": "/u/", "ِ": "/i/" } },
      "diacritics": { "ّ": "doubled", "ْ": "" },
      "phonological_rules": { "al_prefix_moon": { "ipa": "/al-/" }, "al_prefix_sun": { "ipa": "/a-/" } }
    },
    "Modern Standard Arabic (MSA)": {
      "letters": { "ء": { "phoneme": "/ʔ/" }, "ث": { "phoneme": "/θ/" }, "ج": { "phoneme": "/dʒ/" }, "ذ": { "phoneme": "/ð/" }, "ق": { "phoneme": "/q/" }, "ة": { "phoneme": "/a/ or /at/" } },
      "vowels": { "short": { "َ": "/a/", "ُ": "/u/", "ِ": "/i/" } },
      "diacritics": { "ّ": "doubled", "ْ": "" },
      "phonological_rules": { "al_prefix_moon": { "ipa": "/al-/" }, "al_prefix_sun": { "ipa": "/a-/" } }
    }
  }
}

# Sync all dialects with base Egyptian letters
for d in ["Levantine Arabic", "Khaliji Arabic", "Modern Standard Arabic (MSA)"]:
    base = IPA_RULES["dialects"]["Egyptian"]["letters"].copy()
    base.update(IPA_RULES["dialects"][d]["letters"])
    IPA_RULES["dialects"][d]["letters"] = base
    IPA_RULES["dialects"][d]["vowels"] = IPA_RULES["dialects"]["Egyptian"]["vowels"]
    IPA_RULES["dialects"][d]["diacritics"].update({"ً": "/an/", "ٌ": "/un/", "ٍ": "/in/"})

# --- 2. Helper Functions ---

def get_rules(dialect):
    if not isinstance(dialect, str): return IPA_RULES["dialects"]["Modern Standard Arabic (MSA)"]
    if "Egyptian" in dialect: return IPA_RULES["dialects"]["Egyptian"]
    if "Levantine" in dialect: return IPA_RULES["dialects"]["Levantine Arabic"]
    if "Khaliji" in dialect or "Gulf" in dialect: return IPA_RULES["dialects"]["Khaliji Arabic"]
    return IPA_RULES["dialects"]["Modern Standard Arabic (MSA)"]

def transcribe(text, dialect):
    if pd.isna(text) or text == "": return ""
    rules = get_rules(dialect)
    letters = rules["letters"]
    diacritics = {**rules["vowels"]["short"], **rules["diacritics"]}
    sun_letters = set(rules["phonological_rules"]["al_prefix_sun"].get("letters", "تثدذرزسشصضطظلن").replace(",", ""))

    words = str(text).split()
    results = []

    for word in words:
        out = ""
        i = 0
        while i < len(word):
            char = word[i]

            # 1. Handle "li-l-" (للـ) prefix
            if char == 'ل' and i + 1 < len(word) and word[i+1] == 'ل':
                out += "/l/il-/"
                i += 2
                continue

            # 2. Handle "Al-" (الـ) Prefix
            if char == 'ا' and i + 1 < len(word) and word[i+1] == 'ل':
                if i + 2 < len(word) and word[i+2] in sun_letters:
                    prefix = rules["phonological_rules"]["al_prefix_sun"]["ipa"].strip('/')
                    sun_ph = letters.get(word[i+2], {}).get("phoneme", "").strip('/')
                    out += f"/{prefix}{sun_ph}{sun_ph}/"
                    i += 3
                    continue
                else:
                    out += rules["phonological_rules"]["al_prefix_moon"]["ipa"]
                    i += 2
                    continue

            # 3. Handle Letters
            if char in letters:
                ph = letters[char]["phoneme"]
                if " or " in ph: ph = ph.split(" or ")[0] # Pausal form for ة
                out += ph

            # 4. Handle Diacritics/Shadda
            if i + 1 < len(word) and word[i+1] in diacritics:
                d = word[i+1]
                if d == "ّ":
                    # Double the previous sound segment
                    clean_out = out.strip('/')
                    if clean_out:
                        last_unit = clean_out.split('/')[-1]
                        out += f"/{last_unit}/"
                else:
                    out += diacritics[d]
                i += 1
            i += 1

        # Clean up separators for the word
        clean_word = out.replace("//", "/").strip('/')
        if clean_word:
            results.append("/" + clean_word + "/")

    return " ".join(results)

# --- 3. Main Logic ---

def process_metadata_file(file_path):
    """Handles the loading, transcribing, and saving of a single CSV file."""
    try:
        # Load CSV
        try:
            df = pd.read_csv(file_path, sep='|', encoding='utf-8')
        except UnicodeDecodeError:
            df = pd.read_csv(file_path, sep='|', encoding='windows-1256')

        # Flexible Column Mapping
        cols = {c.lower().strip(): c for c in df.columns}

        t_col = cols.get('transcript')
        # Check for 'main dialect' or 'dialect'
        d_col = cols.get('main dialect') or cols.get('dialect')
        # Check for 'phoneme' or 'phonemes'
        p_col = cols.get('phoneme') or cols.get('phonemes') or 'Phoneme'

        if not t_col or not d_col:
            print(f"  ⚠️ Skipping {file_path}: Missing required columns. Found: {list(df.columns)}")
            return False

        # Apply G2P Transcription
        df[p_col] = df.apply(lambda r: transcribe(r[t_col], r[d_col]), axis=1)

        # Save back with same format
        df.to_csv(file_path, index=False, sep='|', encoding='utf-8')
        print(f"  ✅ Success: Transcribed {len(df)} rows.")
        return True

    except Exception as e:
        print(f"  🛑 Error processing {file_path}: {e}")
        return False


if __name__ == "__main__":
    drive.mount('/content/drive')

    # Ask for the main directory instead of a single file
    main_folder_path = input("Enter the main folder path to scan for metadata.csv files: ").strip()

    if not os.path.exists(main_folder_path):
        print(f"Error: The path '{main_folder_path}' does not exist.")
    else:
        print(f"\nScanning for 'metadata.csv' inside: {main_folder_path}")
        files_processed = 0

        # os.walk automatically loops through all subdirectories recursively
        for root, dirs, files in os.walk(main_folder_path):
            if "metadata.csv" in files:
                full_path = os.path.join(root, "metadata.csv")
                print(f"\nFound: {full_path}")

                # Process the file
                success = process_metadata_file(full_path)
                if success:
                    files_processed += 1

        print("\n" + "="*40)
        print(f"Done! Successfully processed {files_processed} files.")
        print("="*40)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Enter the main folder path to scan for metadata.csv files: /content/drive/MyDrive/Masters Arabic TTS/Organized

Scanning for 'metadata.csv' inside: /content/drive/MyDrive/Masters Arabic TTS/Organized

Found: /content/drive/MyDrive/Masters Arabic TTS/Organized/Levantine/Khaled Ghattas/Topic_007/metadata.csv
  ✅ Success: Transcribed 167 rows.

Found: /content/drive/MyDrive/Masters Arabic TTS/Organized/Levantine/Khaled Ghattas/Topic_008/metadata.csv
  ✅ Success: Transcribed 296 rows.

Found: /content/drive/MyDrive/Masters Arabic TTS/Organized/Levantine/Khaled Ghattas/Topic_009/metadata.csv
  ✅ Success: Transcribed 206 rows.

Found: /content/drive/MyDrive/Masters Arabic TTS/Organized/Levantine/Khaled Ghattas/Topic_001/metadata.csv
  ✅ Success: Transcribed 166 rows.

Found: /content/drive/MyDrive/Masters Arabic TTS/Organized/Levantine/Khaled Ghattas/Topic_002/meta